In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_report_unified_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        # "ExtReportUnifiedId string"
        # ",SourceSystem string"
        # ",PortfolioKey string"
        # ",PortfolioName string"
        # ",IngestionTimestamp timestamp"
        # ",BatchId string"
        # ",ExtReportUnifiedKey string"
        # ",AssetSegment string"
        # ",ApplicationBranchCode string"
        # ",BranchManagementDepartment string"
        # ",ApplicationBranch string"
        # ",ApplicationNumber string"
        # ",CommonApplicationStep string"
        # ",AgreementKey string"
        # ",PartyKey string"
        # ",PartyName string"
        # ",PartyTaxIdentificationNumber string"
        # ",FirstApprovalReceivedDate string"
        # ",UserFullName string"
        # ",LastAuthorityLevelDate string"
        # ",LastApprovalReceivedDate string"
        # ",LastApprovalSendDate string"
        # ",LastReevaluateReceivedDate string"
        # ",ReEvaluationTimes string"
        # ",Product string"
        # ",Origination string"
        # ",DEK string"
        # ",ExposureOriginalAmount string"
        # ",ExposureTotalDuration string"
        # ",ExposureKey string"
        # ",TotalOutstandingAmount string"
        # ",Decision string"
        # ",DecisionReason string"
        # ",ApprovalTerms string"
        # ",Currency string"
        # ",NextActions string"
        # ",HraklisAgreementKey string"
        # ",DisbursementCompletion string"
        # ",ImplementationApplicationNumber string"
        # ",ImplemenationApplicationStep string"
        # ",ImplementationDate string"
        # ",RestructuringType string"
        # ",FirstDateReceiptValuation string"
        # ",ApplicationActivationDate string"
        # ",CurrentApplicationStepCode string"
        # ",CurrentApplicationStep string"
        # ",ApplicationActivationBranch string"
        # ",HubCode string"
        # ",Hub string"
        # ",StakodCode string"
        # ",Stakod string"
        # ",AutoCreditFlag string"
        # ",ReturnReason string"
        # ",UnifiedApplicationStatus string"
        # ",NewProduct string"
        # ",Channel string"
        # ",AutoDisbursementDate string"
        # ",Litigious string"
        # ",NPLSolution string"
        # ",ReceiptDateCurrentStep string"
        # ",ProductCode string"
        # ",TeamNumber string"
        # ",AutoCreditAtApplicationLevel string"
        # ",AuthorityLevel string"
        # ",ImplementationApplicationDivision string"
        # ",ImplementationCurrentStepDateReceived string"
        # ",CheckContractualDocuments string"
        # ",AutoPrintFlag string"
        # ",PrintException string"
        # ",ContractPrint string"
        # ",ApplicationAutoPrintFlag string"
        # ",ApplicationPreAppovedFlag string"
        # ",N3869Flag string"
        # ",N3869CommonProduct string"
        # ",RequestAutocredit string"
        # ",ApplicationAutocredit string"
        # ",HasFinancialDifficulties string"
        # ",InitialSignature string"
        # ",ExcludedParticipant string"
        # ",ApprovedExcludedParticipant string"
        # ",ApprovalDifferentTerms string"
        # ",ApplicationApprovalWithDifferentTerms string"
        # ",SignatureRequired string"
        # ",ApplicationSignatureRequired string"
        # ",TotalCreditRiskOverLimit string"
        # ",CurrentStepDate string"
        # ",SignedByBankDate string"
        # ",counterA string"
        # ",counterB string"
        # ",SupplementaryPrenotation string"
        # ",LastDateShippedToBranchDKLT string"
        # ",UserDataValidatorProcessorPerApplication string"
        # ",ResponsibleApprovalAuthorityLevel string"
        # ",ConfirmRegistrationCorrectnessFlag string"
        # ",ConfirmDocumentsCompleteness string"
        # ",ApplicationDynamicChangedFlag string"
        # ",ApprovalConditions string"
        # ",SticValuation string"
        # ",FirstDateSendDecisionToProcessor string"
        # ",LastDateSendDecisionToProcessor string"
        # ",ReValuationFlag string"
        # ",NumberOfReturns string"
        # ",ApprovalSubRole string"
        # ",LegalAction string"
        # ",N4605Flag string"
        # ",RegistratorCode string"
        # ",RegistratorName string"
        # ",FirstTimePrepareDecision string"
        # ",LastUpdatedDate string"
        # ",ExposureDeclarationStandingOrderException string"
        # ",RestructuringAmount string"
        # ",RestrectedDueAmount string"
        # ",DYPRootingGroup string"
        # ",StandaloneImplementation string"
        # ",ImplemenationStatus string"
        # ",ConsiderControlProductionContracts string"
        # ",ConsiderImplementationControl string"
        # ",ResponsibleRegisterNotice string"
        # ",ResponsibleCode string"
        # ",RegisterNotice string"
        # ",AssignedFlag string"
        # ",AssignmentDate string"
        # ",Result string"
        # ",ResultDate string"
        # ",TerminationReason string"
        # ",ApplicationImplementationFlag string"
        # ",ApplicationImplementationDate string"
        # ",Exposures string"
        # ",ImplementationConsideration string"
        # ",GalaxyFlag string"
        # ",GalaxySecuritization string"
        # ",Covid19 string"
        # ",Authority string"
        # ",AuthorityDynamicChangedFlag string"
        # ",BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtReportUnified"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== report_unified_cl_vw ==================

        report_unified_cl_vw_extra_filters = None

        if report_unified_cl_vw_extra_filters is not None:
            report_unified_cl_vw_extra_filters.append(date_filter)
            report_unified_cl_vw_final_filters = " AND ".join(
                report_unified_cl_vw_extra_filters
            )
        else:
            report_unified_cl_vw_final_filters = date_filter

        report_unified_cl_vw_selected_fields = [
            "ApplicationBranch",
            "ApplicationNumber",
            "CommonApplicationStep",
            "CDI",
            "PartyName",
            "PartyTaxIdentificationNumber",
            "FirstApprovalReceivedDate",
            "UserFullName",
            "LastApprovalReceivedDate",
            "LastApprovalSendDate",
            "LastReevaluateReceivedDate",
            "ReEvaluationTimes",
            "Product",
            "Origination",
            "DEK",
            "ExposureOriginalAmount",
            "ExposureTotalDuration",
            "ExposureKey",
            "TotalOutstandingAmount",
            "Decision",
            "DecisionReason",
            "ApprovalTerms",
            "NextActions",
            "HraklisAgreementKey",
            "DisbursementCompletion",
            "ImplementationApplicationNumber",
            "ImplemenationApplicationStep",
            "ImplementationDate",
            "RestructuringType",
            "FirstDateReceiptValuation",
            "ApplicationActivationDate",
            "CurrentApplicationStepCode",
            "ApplicationActivationBranch",
            "HubCode",
            "Hub",
            "AutoCreditFlag",
            "ReturnReason",
            "UnifiedApplicationStatus",
            "NewProduct",
            "Channel",
            "AutoDisbursementDate",
            "Litigious",
            "NPLSolution",
            "ReceiptDateCurrentStep",
            "ProductCode",
            "TeamNumber",
            "AutoCreditAtApplicationLevel",
            "AuthorityLevel",
            "ImplementationApplicationDivision",
            "ImplementationCurrentStepDateReceived",
            "CheckContractualDocuments",
            "AutoPrintFlag",
            "PrintException",
            "ApplicationAutoPrintFlag",
            "ApplicationPreAppovedFlag",
            "L3869Flag",
            "L3869CommonProduct",
            "RequestAutocredit",
            "ApplicationAutocredit",
            "InitialSignature",
            "ExcludedParticipant",
            "ApprovedExcludedParticipant",
            "ApprovalDifferentTerms",
            "ApplicationApprovalWithDifferentTerms",
            "SignatureRequired",
            "ApplicationSignatureRequired",
            "TotalCreditRiskOverLimit",
            "SignedByBankDate",
            "CounterA",
            "CounterΒ",
            "SupplementaryPrenotation",
            "LastDateShippedToBranchDKLT",
            "UserDataValidatorProcessorPerApplication",
            "ResponsibleApprovalAuthorityLevel",
            "ConfirmRegistrationCorrectnessFlag",
            "ConfirmDocumentsCompleteness",
            "ApplicationDynamicChangedFlag",
            "ApprovalConditions",
            "SticValuation",
            "FirstDateSendDecisionToProcessor",
            "LastDateSendDecisionToProcessor",
            "ReValuationFlag",
            "NumberOfReturns",
            "ApprovalSubRole",
            "LegalAction",
            "L4605Flag",
            "RegistratorCode",
            "RegistratorName",
            "FirstTimePrepareDecision",
            "LastUpdatedDate",
            "ExposureDeclarationStandingOrderException",
            "RestructuringAmount",
            "RestrectedDueAmount",
            "DYPRootingGroup",
            "StandaloneImplementation",
            "ImplemenationStatus",
            "ConsiderControlProductionContracts",
            "ConsiderImplementationControl",
            "ResponsibleRegisterNotice",
            "ResponsibleCode",
            "RegisterNotice",
            "AssignedFlag",
            "AssignmentDate",
            "Result",
            "ResultDate",
            "TerminationReason",
            "ApplicationImplementationFlag",
            "ApplicationImplementationDate",
            "Exposures",
            "ImplementationConsideration",
            "GalaxyFlag",
            "GalaxySecuritization",
            "Covid19",
            "Authority",
            "AuthorityDynamicChangedFlag",
            "BusinessDate",
        ]

        report_unified_cl_vw = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extreport_unified_cl",
            report_unified_cl_vw_final_filters,
            report_unified_cl_vw_selected_fields,
        )

        # ================== report_unified_ml_vw ==================

        report_unified_ml_vw_extra_filters = None

        if report_unified_ml_vw_extra_filters is not None:
            report_unified_ml_vw_extra_filters.append(date_filter)
            report_unified_ml_vw_final_filters = " AND ".join(
                report_unified_ml_vw_extra_filters
            )
        else:
            report_unified_ml_vw_final_filters = date_filter

        report_unified_ml_vw_selected_fields = [
            "ApplicationBranch",
            "ApplicationNumber",
            "PartyName",
            "PartyTaxIdentificationNumber",
            "FirstApprovalReceivedDate",
            "LastApprovalUser",
            "LastApprovalReceivedDate",
            "LastApprovalSendDate",
            "LastReevaluateReceivedDate",
            "ReEvaluationTimes",
            "ProductType",
            "RequestType",
            "DEK",
            "ExposureKey",
            "Balance",
            "ApprovalTerms",
            "NextActions",
            "AgreementKey",
            "ImplemenationApplicationStep",
            "ImplementationDate",
            "StepId",
            "NplHub",
            "Hub",
            "AutoCreditFlag",
            "ReturnReason",
            "Channel",
            "Denounced",
            "CommonApplicationStep",
            "ImplementationCurrentStepDateReceived",
            "AutoPrintFlag",
            "PrintException",
            "ApplicationAutoPrintFlag",
            "ContractPrint",
            "L3869Flag",
            "L3869Commonprod",
            "RequestAutocredit",
            "ApplicationAutocredit",
            "InitialSignature",
            "ExcludedParticipant",
            "ApprovedExcludedParticipant",
            "ApprovalDifferentTerms",
            "ApplicationApprovalWithDifferentTerms",
            "SignatureRequired",
            "ApplicationSignatureRequired",
            "TotalCreditRiskOverLimit",
            "SignedByBankDate",
            "counterA",
            "counterB",
            "LastDateShippedToBranchDKLT",
            "UserDataValidatorProcessorPerApplication",
            "ResponsibleApprovalAuthorityLevel",
            "ConfirmRegistrationCorrectnessFlag",
            "ConfirmDocumentsCompleteness",
            "ApprovalConditions",
            "SticValuation",
            "FirstDateSendDecisionToProcessor",
            "LastDateSendDecisionToProcessor",
            "NumberOfReturns",
            "ApprovalSubRole",
            "DueAmount",
            "Decision",
            "AuthorityDynamicChangedFlag",
            "UnifiedApplicationStatus",
            "FirstDateReceiptValuation",
            "BusinessDate",
        ]

        report_unified_ml_vw = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extreport_unified_ml",
            report_unified_ml_vw_final_filters,
            report_unified_ml_vw_selected_fields,
        )

        # ================== report_unified_sb_vw ==================

        report_unified_sb_vw_extra_filters = None

        if report_unified_sb_vw_extra_filters is not None:
            report_unified_sb_vw_extra_filters.append(date_filter)
            report_unified_sb_vw_final_filters = " AND ".join(
                report_unified_sb_vw_extra_filters
            )
        else:
            report_unified_sb_vw_final_filters = date_filter

        report_unified_sb_vw_selected_fields = [
            "BranchName",
            "ApplicationNumber",
            "CDI",
            "PartyName",
            "PartyTaxIdentificationNumber",
            "ApprovalCenterDate",
            "CounselUser",
            "LastDateShippedToBranchDKLT",
            "ProductType",
            "DEK",
            "ExposureKey",
            "SticValuation",
            "Desicion",
            "DecisionReason",
            "NextActions",
            "ImplementationApplicationNumber",
            "ResrtucturingType",
            "FirstDateReceiptValuation",
            "DateStarted",
            "StepId",
            "ApplicationBranch",
            "NPLHubs",
            "AutoCreditFlag",
            "ReturnReason",
            "UnifiedApplicationStatus",
            "Channel",
            "Denounced",
            "NPLSolution",
            "CurrentStepReceiptDate",
            "AutoCreditAtApplicationLevel",
            "AutoPrintFlag",
            "PrintException",
            "ApplicationAutoPrintFlag",
            "ApplicationPreAppovedFlag",
            "L3869Flag",
            "L3869Commonprod",
            "RequestAutocredit",
            "ApplicationAutocredit",
            "InitialSignature",
            "ExcludedParticipant",
            "ApprovedExcludedParticipant",
            "ApprovalDifferentTerms",
            "ApplicationApprovalWithDifferentTerms",
            "SignatureRequired",
            "ApplicationSignatureRequired",
            "TotalCreditRiskOverLimit",
            "HasFinancialDifficulties",
            "CurrentStepInsertionDate",
            "SignedByBankDate",
            "CounterA",
            "CounterΒ",
            "UserDataValidatorProcessorPerApplication",
            "ApprovalLevel",
            "AuthorityLevel",
            "ConfirmRegistrationCorrectnessFlag",
            "ConfirmDocumentsCompleteness",
            "ApplicationDynamicChangedFlag",
            "ApprovalConditions",
            "FirstDateSendDecisionToProcessor",
            "LastDateSendDecisionToProcessor",
            "ReValuationFlag",
            "NumberOfReturns",
            "ApprovalSubRole",
            "SuggestedAmount",
            "ImplementationReason",
            "Implementation",
            "Stakod",
            "StakodDescription",
            "Description",
            "Currency",
            "ContractNo",
            "LastAuthorityLevelDate",
            "ApplicationActivationBranch",
            "BranchManagmentDepartment",
            "BusinessDate",
        ]

        report_unified_sb_vw = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extreport_unified_sb",
            report_unified_sb_vw_final_filters,
            report_unified_sb_vw_selected_fields,
        )

        # =============== result_df_cl   ========================

        # HashKeys
        target_table_cl_primary_key_fields = [
            report_unified_cl_vw.ApplicationBranch,
            report_unified_cl_vw.ApplicationNumber,
            report_unified_cl_vw.ExposureKey,
            lit("extfile"),
        ]

        # Target table(hash keys and calculations)
        target_table_cl = (
            report_unified_cl_vw.withColumn(
                "ExtReportUnifiedId",
                sha2(
                    concat_ws(";", *target_table_cl_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExtReportUnifiedKey",
                concat_ws(
                    ";",
                    report_unified_cl_vw.ApplicationBranch,
                    report_unified_cl_vw.ApplicationNumber,
                    report_unified_cl_vw.ExposureKey,
                ),
            )
            .withColumn(
                "PortfolioKey",
                lit("-9999"),
            )
            .withColumn(
                "PortfolioName",
                lit("Core Retail"),
            )
        )

        result_df_cl = target_table_cl.select(
            target_table_cl.ExtReportUnifiedId.alias("ExtReportUnifiedId"),
            lit("extfile").alias("SourceSystem"),
            target_table_cl.PortfolioKey.alias("PortfolioKey"),
            target_table_cl.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_cl.ExtReportUnifiedKey.alias("ExtReportUnifiedKey"),
            lit("CL").alias("AssetSegment"),
            lit(None).alias("ApplicationBranchCode"),
            lit(None).alias("BranchManagementDepartment"),
            target_table_cl.ApplicationBranch.alias("ApplicationBranch"),
            target_table_cl.ApplicationNumber.alias("ApplicationNumber"),
            target_table_cl.CommonApplicationStep.alias("CommonApplicationStep"),
            lit(None).alias("AgreementKey"),
            target_table_cl.CDI.alias("PartyKey"),
            target_table_cl.PartyName.alias("PartyName"),
            target_table_cl.PartyTaxIdentificationNumber.alias("PartyTaxIdentificationNumber"),
            target_table_cl.FirstApprovalReceivedDate.alias("FirstApprovalReceivedDate"),
            target_table_cl.UserFullName.alias("UserFullName"),
            lit(None).alias("LastAuthorityLevelDate"),
            target_table_cl.LastApprovalReceivedDate.alias("LastApprovalReceivedDate"),
            report_unified_cl_vw.LastApprovalSendDate.alias("LastApprovalSendDate"),
            report_unified_cl_vw.LastReevaluateReceivedDate.alias("LastReevaluateReceivedDate"),
            report_unified_cl_vw.ReEvaluationTimes.alias("ReEvaluationTimes"),
            target_table_cl.Product.alias("Product"),
            report_unified_cl_vw.Origination.alias("Origination"),
            target_table_cl.DEK.alias("DEK"),
            report_unified_cl_vw.ExposureOriginalAmount.alias("ExposureOriginalAmount"),
            report_unified_cl_vw.ExposureTotalDuration.alias("ExposureTotalDuration"),
            target_table_cl.ExposureKey.alias("ExposureKey"),
            target_table_cl.TotalOutstandingAmount.alias("TotalOutstandingAmount"),
            target_table_cl.Decision.alias("Decision"),
            target_table_cl.DecisionReason.alias("DecisionReason"),
            report_unified_cl_vw.ApprovalTerms.alias("ApprovalTerms"),
            lit(None).alias("Currency"),
            target_table_cl.NextActions.alias("NextActions"),
            report_unified_cl_vw.HraklisAgreementKey.alias("HraklisAgreementKey"),
            report_unified_cl_vw.DisbursementCompletion.alias("DisbursementCompletion"),
            target_table_cl.ImplementationApplicationNumber.alias(
                "ImplementationApplicationNumber"
            ),
            report_unified_cl_vw.ImplemenationApplicationStep.alias(
                "ImplemenationApplicationStep"
            ),
            report_unified_cl_vw.ImplementationDate.alias("ImplementationDate"),
            target_table_cl.RestructuringType.alias("RestructuringType"),
            report_unified_cl_vw.FirstDateReceiptValuation.alias(
                "FirstDateReceiptValuation"
            ),
            target_table_cl.ApplicationActivationDate.alias(
                "ApplicationActivationDate"
            ),
            target_table_cl.CurrentApplicationStepCode.alias(
                "CurrentApplicationStepCode"
            ),
            target_table_cl.CurrentApplicationStepCode.alias(
                "CurrentApplicationStep"
            ),
            target_table_cl.ApplicationActivationBranch.alias(
                "ApplicationActivationBranch"
            ),
            target_table_cl.HubCode.alias("HubCode"),
            target_table_cl.Hub.alias("Hub"),
            lit(None).alias("StakodCode"),
            lit(None).alias("Stakod"),
            target_table_cl.AutoCreditFlag.alias("AutoCreditFlag"),
            target_table_cl.ReturnReason.alias("ReturnReason"),
            target_table_cl.UnifiedApplicationStatus.alias("UnifiedApplicationStatus"),
            report_unified_cl_vw.NewProduct.alias("NewProduct"),
            report_unified_cl_vw.Channel.alias("Channel"),
            report_unified_cl_vw.AutoDisbursementDate.alias("AutoDisbursementDate"),
            target_table_cl.Litigious.alias("Litigious"),
            target_table_cl.NPLSolution.alias("NPLSolution"),
            target_table_cl.ReceiptDateCurrentStep.alias("ReceiptDateCurrentStep"),
            report_unified_cl_vw.ProductCode.alias("ProductCode"),
            report_unified_cl_vw.TeamNumber.alias("TeamNumber"),
            target_table_cl.AutoCreditAtApplicationLevel.alias(
                "AutoCreditAtApplicationLevel"
            ),
            target_table_cl.AuthorityLevel.alias("AuthorityLevel"),
            report_unified_cl_vw.ImplementationApplicationDivision.alias(
                "ImplementationApplicationDivision"
            ),
            report_unified_cl_vw.ImplementationCurrentStepDateReceived.alias(
                "ImplementationCurrentStepDateReceived"
            ),
            report_unified_cl_vw.CheckContractualDocuments.alias(
                "CheckContractualDocuments"
            ),
            target_table_cl.AutoPrintFlag.alias("AutoPrintFlag"),
            target_table_cl.PrintException.alias("PrintException"),
            lit(None).alias("ContractPrint"),
            target_table_cl.ApplicationAutoPrintFlag.alias(
                "ApplicationAutoPrintFlag"
            ),
            target_table_cl.ApplicationPreAppovedFlag.alias(
                "ApplicationPreAppovedFlag"
            ),
            target_table_cl.L3869Flag.alias("N3869Flag"),
            target_table_cl.L3869CommonProduct.alias("N3869CommonProduct"),
            target_table_cl.RequestAutocredit.alias("RequestAutocredit"),
            target_table_cl.ApplicationAutocredit.alias("ApplicationAutocredit"),
            lit(None).alias("HasFinancialDifficulties"),
            target_table_cl.InitialSignature.alias("InitialSignature"),
            target_table_cl.ExcludedParticipant.alias("ExcludedParticipant"),
            target_table_cl.ApprovedExcludedParticipant.alias(
                "ApprovedExcludedParticipant"
            ),
            target_table_cl.ApprovalDifferentTerms.alias("ApprovalDifferentTerms"),
            target_table_cl.ApplicationApprovalWithDifferentTerms.alias(
                "ApplicationApprovalWithDifferentTerms"
            ),
            target_table_cl.SignatureRequired.alias("SignatureRequired"),
            target_table_cl.ApplicationSignatureRequired.alias(
                "ApplicationSignatureRequired"
            ),
            target_table_cl.TotalCreditRiskOverLimit.alias(
                "TotalCreditRiskOverLimit"
            ),
            lit(None).alias("CurrentStepDate"),
            target_table_cl.SignedByBankDate.alias("SignedByBankDate"),
            target_table_cl.CounterA.alias("counterA"),
            target_table_cl.CounterΒ.alias("counterB"),
            target_table_cl.SupplementaryPrenotation.alias("SupplementaryPrenotation"),
            target_table_cl.LastDateShippedToBranchDKLT.alias(
                "LastDateShippedToBranchDKLT"
            ),
            target_table_cl.UserDataValidatorProcessorPerApplication.alias(
                "UserDataValidatorProcessorPerApplication"
            ),
            target_table_cl.ResponsibleApprovalAuthorityLevel.alias(
                "ResponsibleApprovalAuthorityLevel"
            ),
            target_table_cl.ConfirmRegistrationCorrectnessFlag.alias(
                "ConfirmRegistrationCorrectnessFlag"
            ),
            target_table_cl.ConfirmDocumentsCompleteness.alias(
                "ConfirmDocumentsCompleteness"
            ),
            target_table_cl.ApplicationDynamicChangedFlag.alias(
                "ApplicationDynamicChangedFlag"
            ),
            target_table_cl.ApprovalConditions.alias("ApprovalConditions"),
            lit(None).alias("SticValuation"),
            target_table_cl.FirstDateSendDecisionToProcessor.alias(
                "FirstDateSendDecisionToProcessor"
            ),
            target_table_cl.LastDateSendDecisionToProcessor.alias(
                "LastDateSendDecisionToProcessor"
            ),
            target_table_cl.ReValuationFlag.alias("ReValuationFlag"),
            target_table_cl.NumberOfReturns.alias("NumberOfReturns"),
            target_table_cl.ApprovalSubRole.alias("ApprovalSubRole"),
            target_table_cl.LegalAction.alias("LegalAction"),
            target_table_cl.L4605Flag.alias("N4605Flag"),
            target_table_cl.RegistratorCode.alias("RegistratorCode"),
            target_table_cl.RegistratorName.alias("RegistratorName"),
            target_table_cl.FirstTimePrepareDecision.alias(
                "FirstTimePrepareDecision"
            ),
            target_table_cl.LastUpdatedDate.alias("LastUpdatedDate"),
            target_table_cl.ExposureDeclarationStandingOrderException.alias(
                "ExposureDeclarationStandingOrderException"
            ),
            target_table_cl.RestructuringAmount.alias("RestructuringAmount"),
            target_table_cl.RestrectedDueAmount.alias("RestrectedDueAmount"),
            target_table_cl.DYPRootingGroup.alias("DYPRootingGroup"),
            target_table_cl.StandaloneImplementation.alias(
                "StandaloneImplementation"
            ),
            target_table_cl.ImplemenationStatus.alias("ImplemenationStatus"),
            target_table_cl.ConsiderControlProductionContracts.alias(
                "ConsiderControlProductionContracts"
            ),
            target_table_cl.ConsiderImplementationControl.alias(
                "ConsiderImplementationControl"
            ),
            target_table_cl.ResponsibleRegisterNotice.alias(
                "ResponsibleRegisterNotice"
            ),
            target_table_cl.ResponsibleCode.alias("ResponsibleCode"),
            target_table_cl.RegisterNotice.alias("RegisterNotice"),
            target_table_cl.AssignedFlag.alias("AssignedFlag"),
            target_table_cl.AssignmentDate.alias("AssignmentDate"),
            target_table_cl.Result.alias("Result"),
            target_table_cl.ResultDate.alias("ResultDate"),
            target_table_cl.TerminationReason.alias("TerminationReason"),
            target_table_cl.ApplicationImplementationFlag.alias(
                "ApplicationImplementationFlag"
            ),
            target_table_cl.ApplicationImplementationDate.alias(
                "ApplicationImplementationDate"
            ),
            target_table_cl.Exposures.alias("Exposures"),
            target_table_cl.ImplementationConsideration.alias(
                "ImplementationConsideration"
            ),
            target_table_cl.GalaxyFlag.alias("GalaxyFlag"),
            target_table_cl.GalaxySecuritization.alias("GalaxySecuritization"),
            target_table_cl.Covid19.alias("Covid19"),
            target_table_cl.Authority.alias("Authority"),
            target_table_cl.AuthorityDynamicChangedFlag.alias(
                "AuthorityDynamicChangedFlag"
            ),
            target_table_cl.BusinessDate.alias("BusinessDate"),
        )


        # ======================== result_df_ml =====================

        # HashKeys
        target_table_ml_primary_key_fields = [
            report_unified_ml_vw.ApplicationBranch,
            report_unified_ml_vw.ApplicationNumber,
            report_unified_ml_vw.ExposureKey,
            lit("extfile"),
        ]

        target_table_ml = (
            report_unified_ml_vw.withColumn(
                "ExtReportUnifiedId",
                sha2(
                    concat_ws(";", *target_table_ml_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExtReportUnifiedKey",
                concat_ws(
                    ";",
                    report_unified_ml_vw.ApplicationBranch,
                    report_unified_ml_vw.ApplicationNumber,
                    report_unified_ml_vw.ExposureKey,
                ),
            )
            .withColumn(
                "PortfolioKey",
                lit("-9999"),
            )
            .withColumn(
                "PortfolioName",
                lit("Core Retail"),
            )
        )

        result_df_ml = target_table_ml.select(
            target_table_ml.ExtReportUnifiedId.alias("ExtReportUnifiedId"),
            lit("extfile").alias("SourceSystem"),
            target_table_ml.PortfolioKey.alias("PortfolioKey"),
            target_table_ml.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_ml.ExtReportUnifiedKey.alias("ExtReportUnifiedKey"),
            lit("ML").alias("AssetSegment"),
            lit(None).alias("ApplicationBranchCode"),
            lit(None).alias("BranchManagementDepartment"),
            target_table_ml.ApplicationBranch.alias("ApplicationBranch"),
            target_table_ml.ApplicationNumber.alias("ApplicationNumber"),
            lit(None).alias("CommonApplicationStep"),
            target_table_ml.AgreementKey.alias("AgreementKey"),
            lit(None).alias("PartyKey"),
            target_table_ml.PartyName.alias("PartyName"),
            target_table_ml.PartyTaxIdentificationNumber.alias("PartyTaxIdentificationNumber"),
            target_table_ml.FirstApprovalReceivedDate.alias(
                "FirstApprovalReceivedDate"
            ),
            target_table_ml.LastApprovalUser.alias("UserFullName"),
            lit(None).alias("LastAuthorityLevelDate"),
            target_table_ml.LastApprovalReceivedDate.alias(
                "LastApprovalReceivedDate"
            ),
            report_unified_ml_vw.LastApprovalSendDate.alias("LastApprovalSendDate"),
            report_unified_ml_vw.LastReevaluateReceivedDate.alias(
                "LastReevaluateReceivedDate"
            ),
            report_unified_ml_vw.ReEvaluationTimes.alias("ReEvaluationTimes"),
            target_table_ml.ProductType.alias("Product"),
            lit(None).alias("Origination"),
            target_table_ml.DEK.alias("DEK"),
            lit(None).alias("ExposureOriginalAmount"),
            lit(None).alias("ExposureTotalDuration"),
            target_table_ml.ExposureKey.alias("ExposureKey"),
            target_table_ml.Balance.alias("TotalOutstandingAmount"),
            report_unified_ml_vw.Decision.alias("Decision"),
            lit(None).alias("DecisionReason"),
            report_unified_ml_vw.ApprovalTerms.alias("ApprovalTerms"),
            lit(None).alias("Currency"),
            target_table_ml.NextActions.alias("NextActions"),
            lit(None).alias("HraklisAgreementKey"),
            lit(None).alias("DisbursementCompletion"),
            lit(None).alias("ImplementationApplicationNumber"),
            report_unified_ml_vw.ImplemenationApplicationStep.alias(
                "ImplemenationApplicationStep"
            ),
            report_unified_ml_vw.ImplementationDate.alias(
                "ImplementationDate"
            ),
            lit(None).alias("RestructuringType"),
            report_unified_ml_vw.FirstDateReceiptValuation.alias("FirstDateReceiptValuation"),
            lit(None).alias("ApplicationActivationDate"),
            target_table_ml.StepId.alias("CurrentApplicationStepCode"),
            target_table_ml.CommonApplicationStep.alias("CurrentApplicationStep"),
            lit(None).alias("ApplicationActivationBranch"),
            target_table_ml.NplHub.alias("HubCode"),
            target_table_ml.Hub.alias("Hub"),
            lit(None).alias("StakodCode"),
            lit(None).alias("Stakod"),
            target_table_ml.AutoCreditFlag.alias("AutoCreditFlag"),
            target_table_ml.ReturnReason.alias("ReturnReason"),
            report_unified_ml_vw.UnifiedApplicationStatus.alias("UnifiedApplicationStatus"),
            lit(None).alias("NewProduct"),
            report_unified_ml_vw.Channel.alias("Channel"),
            lit(None).alias("AutoDisbursementDate"),
            target_table_ml.Denounced.alias("Litigious"),
            lit(None).alias("NPLSolution"),
            lit(None).alias("ReceiptDateCurrentStep"),
            lit(None).alias("ProductCode"),
            lit(None).alias("TeamNumber"),
            lit(None).alias("AutoCreditAtApplicationLevel"),
            lit(None).alias("AuthorityLevel"),
            lit(None).alias("ImplementationApplicationDivision"),
            report_unified_ml_vw.ImplementationCurrentStepDateReceived.alias(
                "ImplementationCurrentStepDateReceived"
            ),
            lit(None).alias("CheckContractualDocuments"),
            target_table_ml.AutoPrintFlag.alias("AutoPrintFlag"),
            target_table_ml.PrintException.alias("PrintException"),
            target_table_ml.ContractPrint.alias("ContractPrint"),
            target_table_ml.ApplicationAutoPrintFlag.alias("ApplicationAutoPrintFlag"),
            lit(None).alias("ApplicationPreAppovedFlag"),
            target_table_ml.L3869Flag.alias("N3869Flag"),
            target_table_ml.L3869Commonprod.alias("N3869CommonProduct"),
            target_table_ml.RequestAutocredit.alias("RequestAutocredit"),
            target_table_ml.ApplicationAutocredit.alias("ApplicationAutocredit"),
            lit(None).alias("HasFinancialDifficulties"),
            target_table_ml.InitialSignature.alias("InitialSignature"),
            target_table_ml.ExcludedParticipant.alias("ExcludedParticipant"),
            target_table_ml.ApprovedExcludedParticipant.alias(
                "ApprovedExcludedParticipant"
            ),
            target_table_ml.ApprovalDifferentTerms.alias("ApprovalDifferentTerms"),
            target_table_ml.ApplicationApprovalWithDifferentTerms.alias(
                "ApplicationApprovalWithDifferentTerms"
            ),
            target_table_ml.SignatureRequired.alias("SignatureRequired"),
            target_table_ml.ApplicationSignatureRequired.alias(
                "ApplicationSignatureRequired"
            ),
            target_table_ml.TotalCreditRiskOverLimit.alias(
                "TotalCreditRiskOverLimit"
            ),
            lit(None).alias("CurrentStepDate"),
            target_table_ml.SignedByBankDate.alias("SignedByBankDate"),
            target_table_ml.counterA.alias("counterA"),
            target_table_ml.counterB.alias("counterB"),
            lit(None).alias("SupplementaryPrenotation"),
            target_table_ml.LastDateShippedToBranchDKLT.alias(
                "LastDateShippedToBranchDKLT"
            ),
            target_table_ml.UserDataValidatorProcessorPerApplication.alias(
                "UserDataValidatorProcessorPerApplication"
            ),
            target_table_ml.ResponsibleApprovalAuthorityLevel.alias(
                "ResponsibleApprovalAuthorityLevel"
            ),
            target_table_ml.ConfirmRegistrationCorrectnessFlag.alias(
                "ConfirmRegistrationCorrectnessFlag"
            ),
            target_table_ml.ConfirmDocumentsCompleteness.alias(
                "ConfirmDocumentsCompleteness"
            ),
            lit(None).alias("ApplicationDynamicChangedFlag"),
            target_table_ml.ApprovalConditions.alias("ApprovalConditions"),
            target_table_ml.SticValuation.alias("SticValuation"),
            target_table_ml.FirstDateSendDecisionToProcessor.alias(
                "FirstDateSendDecisionToProcessor"
            ),
            target_table_ml.LastDateSendDecisionToProcessor.alias(
                "LastDateSendDecisionToProcessor"
            ),
            lit(None).alias("ReValuationFlag"),
            target_table_ml.NumberOfReturns.alias("NumberOfReturns"),
            target_table_ml.ApprovalSubRole.alias("ApprovalSubRole"),
            lit(None).alias("LegalAction"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("RegistratorCode"),
            lit(None).alias("RegistratorName"),
            lit(None).alias("FirstTimePrepareDecision"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("ExposureDeclarationStandingOrderException"),
            lit(None).alias("RestructuringAmount"),
            target_table_ml.DueAmount.alias("RestrectedDueAmount"),
            lit(None).alias("DYPRootingGroup"),
            lit(None).alias("StandaloneImplementation"),
            lit(None).alias("ImplemenationStatus"),
            lit(None).alias("ConsiderControlProductionContracts"),
            lit(None).alias("ConsiderImplementationControl"),
            lit(None).alias("ResponsibleRegisterNotice"),
            lit(None).alias("ResponsibleCode"),
            lit(None).alias("RegisterNotice"),
            lit(None).alias("AssignedFlag"),
            lit(None).alias("AssignmentDate"),
            lit(None).alias("Result"),
            lit(None).alias("ResultDate"),
            lit(None).alias("TerminationReason"),
            lit(None).alias("ApplicationImplementationFlag"),
            lit(None).alias("ApplicationImplementationDate"),
            lit(None).alias("Exposures"),
            lit(None).alias("ImplementationConsideration"),
            lit(None).alias("GalaxyFlag"),
            lit(None).alias("GalaxySecuritization"),
            lit(None).alias("Covid19"),
            lit(None).alias("Authority"),
            target_table_ml.AuthorityDynamicChangedFlag.alias(
                "AuthorityDynamicChangedFlag"
            ),
            target_table_ml.BusinessDate.alias("BusinessDate"),
        )

        # ========================  result_df_sb ====================================

        # HashKeys
        target_table_sb_primary_key_fields = [
            report_unified_sb_vw.BranchName,
            report_unified_sb_vw.ApplicationNumber,
            report_unified_sb_vw.ExposureKey,
            lit("extfile"),
        ]

        # Target table(hash keys and calculations)
        target_table_sb = (
            report_unified_sb_vw.withColumn(
                "ExtReportUnifiedId",
                sha2(
                    concat_ws(";", *target_table_sb_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExtReportUnifiedKey",
                concat_ws(
                    ";",
                    report_unified_sb_vw.BranchName,
                    report_unified_sb_vw.ApplicationNumber,
                    report_unified_sb_vw.ExposureKey,
                ),
            )
            .withColumn(
                "PortfolioKey",
                lit("-9999"),
            )
            .withColumn(
                "PortfolioName",
                lit("Core Retail"),
            )
            .withColumn(
                "Result",
                concat_ws(
                    ";", report_unified_sb_vw.ImplementationReason, report_unified_sb_vw.Implementation
                ),
            )
        )

        # ===================== target_table_sb =====================

        result_df_sb = target_table_sb.select(
            target_table_sb.ExtReportUnifiedId.alias("ExtReportUnifiedId"),
            lit("extfile").alias("SourceSystem"),
            target_table_sb.PortfolioKey.alias("PortfolioKey"),
            target_table_sb.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_sb.ExtReportUnifiedKey.alias("ExtReportUnifiedKey"),
            lit("SB").alias("AssetSegment"),
            target_table_sb.ApplicationActivationBranch.alias("ApplicationBranchCode"),
            target_table_sb.BranchManagmentDepartment.alias(
                "BranchManagementDepartment"
            ),
            target_table_sb.BranchName.alias("ApplicationBranch"),
            target_table_sb.ApplicationNumber.alias("ApplicationNumber"),
            lit(None).alias("CommonApplicationStep"),
            target_table_sb.ContractNo.alias("AgreementKey"),
            target_table_sb.CDI.alias("PartyKey"),
            target_table_sb.PartyName.alias("PartyName"),
            target_table_sb.PartyTaxIdentificationNumber.alias("PartyTaxIdentificationNumber"),
            target_table_sb.ApprovalCenterDate.alias("FirstApprovalReceivedDate"),
            target_table_sb.CounselUser.alias("UserFullName"),
            target_table_sb.LastAuthorityLevelDate.alias("LastAuthorityLevelDate"),
            target_table_sb.LastDateShippedToBranchDKLT.alias(
                "LastApprovalReceivedDate"
            ),
            lit(None).alias("LastApprovalSendDate"),
            lit(None).alias("LastReevaluateReceivedDate"),
            lit(None).alias("ReEvaluationTimes"),
            target_table_sb.ProductType.alias("Product"),
            lit(None).alias("Origination"),
            target_table_sb.DEK.alias("DEK"),
            lit(None).alias("ExposureOriginalAmount"),
            lit(None).alias("ExposureTotalDuration"),
            target_table_sb.ExposureKey.alias("ExposureKey"),
            target_table_sb.SticValuation.alias("TotalOutstandingAmount"),
            target_table_sb.Desicion.alias("Decision"),
            target_table_sb.DecisionReason.alias("DecisionReason"),
            lit(None).alias("ApprovalTerms"),
            target_table_sb.Currency.alias("Currency"),
            target_table_sb.NextActions.alias("NextActions"),
            lit(None).alias("HraklisAgreementKey"),
            lit(None).alias("DisbursementCompletion"),
            target_table_sb.ImplementationApplicationNumber.alias(
                "ImplementationApplicationNumber"
            ),
            lit(None).alias("ImplemenationApplicationStep"),
            lit(None).alias("ImplementationDate"),
            target_table_sb.ResrtucturingType.alias("RestructuringType"),
            report_unified_sb_vw["FirstDateReceiptValuation"].alias(
                "FirstDateReceiptValuation"
            ),
            target_table_sb.DateStarted.alias("ApplicationActivationDate"),
            target_table_sb.StepId.alias("CurrentApplicationStepCode"),
            target_table_sb.Description.alias("CurrentApplicationStep"),
            target_table_sb.ApplicationBranch.alias("ApplicationActivationBranch"),
            target_table_sb.NPLHubs.alias("HubCode"),
            target_table_sb.NPLHubs.alias("Hub"),
            target_table_sb.Stakod.alias("StakodCode"),
            target_table_sb.StakodDescription.alias("Stakod"),
            target_table_sb["AutoCreditFlag"].alias("AutoCreditFlag"),
            target_table_sb.ReturnReason.alias("ReturnReason"),
            target_table_sb.UnifiedApplicationStatus.alias("UnifiedApplicationStatus"),
            lit(None).alias("NewProduct"),
            target_table_sb.Channel.alias("Channel"),
            lit(None).alias("AutoDisbursementDate"),
            target_table_sb.Denounced.alias("Litigious"),
            target_table_sb.NPLSolution.alias("NPLSolution"),
            target_table_sb.CurrentStepReceiptDate.alias("ReceiptDateCurrentStep"),
            lit(None).alias("ProductCode"),
            lit(None).alias("TeamNumber"),
            target_table_sb.AutoCreditAtApplicationLevel.alias(
                "AutoCreditAtApplicationLevel"
            ),
            target_table_sb.AuthorityLevel.alias("AuthorityLevel"),
            lit(None).alias("ImplementationApplicationDivision"),
            lit(None).alias("ImplementationCurrentStepDateReceived"),
            lit(None).alias("CheckContractualDocuments"),
            target_table_sb["AutoPrintFlag"].alias(
                "AutoPrintFlag"
            ),
            target_table_sb["PrintException"].alias(
                "PrintException"
            ),
            lit(None).alias("ContractPrint"),
            target_table_sb.ApplicationAutoPrintFlag.alias(
                "ApplicationAutoPrintFlag"
            ),
            target_table_sb.ApplicationPreAppovedFlag.alias(
                "ApplicationPreAppovedFlag"
            ),
            target_table_sb.L3869Flag.alias("N3869Flag"),
            target_table_sb.L3869Commonprod.alias("N3869CommonProduct"),
            target_table_sb.RequestAutocredit.alias("RequestAutocredit"),
            target_table_sb.ApplicationAutocredit.alias("ApplicationAutocredit"),
            target_table_sb.HasFinancialDifficulties.alias(
                "HasFinancialDifficulties"
            ),
            target_table_sb.InitialSignature.alias("InitialSignature"),
            target_table_sb.ExcludedParticipant.alias("ExcludedParticipant"),
            target_table_sb.ApprovedExcludedParticipant.alias(
                "ApprovedExcludedParticipant"
            ),
            target_table_sb.ApprovalDifferentTerms.alias("ApprovalDifferentTerms"),
            target_table_sb.ApplicationApprovalWithDifferentTerms.alias(
                "ApplicationApprovalWithDifferentTerms"
            ),
            target_table_sb.SignatureRequired.alias("SignatureRequired"),
            target_table_sb.ApplicationSignatureRequired.alias(
                "ApplicationSignatureRequired"
            ),
            target_table_sb.TotalCreditRiskOverLimit.alias(
                "TotalCreditRiskOverLimit"
            ),
            target_table_sb.CurrentStepInsertionDate.alias("CurrentStepDate"),
            target_table_sb.SignedByBankDate.alias("SignedByBankDate"),
            target_table_sb.CounterA.alias("counterA"),
            target_table_sb.CounterΒ.alias("counterB"),
            lit(None).alias("SupplementaryPrenotation"),
            lit(None).alias("LastDateShippedToBranchDKLT"),
            target_table_sb.UserDataValidatorProcessorPerApplication.alias(
                "UserDataValidatorProcessorPerApplication"
            ),
            target_table_sb.ApprovalLevel.alias(
                "ResponsibleApprovalAuthorityLevel"
            ),
            target_table_sb.ConfirmRegistrationCorrectnessFlag.alias(
                "ConfirmRegistrationCorrectnessFlag"
            ),
            target_table_sb.ConfirmDocumentsCompleteness.alias(
                "ConfirmDocumentsCompleteness"
            ),
            target_table_sb.ApplicationDynamicChangedFlag.alias(
                "ApplicationDynamicChangedFlag"
            ),
            target_table_sb.ApprovalConditions.alias("ApprovalConditions"),
            lit(None).alias("SticValuation"),
            target_table_sb.FirstDateSendDecisionToProcessor.alias(
                "FirstDateSendDecisionToProcessor"
            ),
            target_table_sb.LastDateSendDecisionToProcessor.alias(
                "LastDateSendDecisionToProcessor"
            ),
            target_table_sb.ReValuationFlag.alias("ReValuationFlag"),
            target_table_sb.NumberOfReturns.alias("NumberOfReturns"),
            target_table_sb.ApprovalSubRole.alias("ApprovalSubRole"),
            lit(None).alias("LegalAction"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("RegistratorCode"),
            lit(None).alias("RegistratorName"),
            lit(None).alias("FirstTimePrepareDecision"),
            lit(None).alias("LastUpdatedDate"),
            lit(None).alias("ExposureDeclarationStandingOrderException"),
            target_table_sb.SuggestedAmount.alias("RestructuringAmount"),
            lit(None).alias("RestrectedDueAmount"),
            lit(None).alias("DYPRootingGroup"),
            lit(None).alias("StandaloneImplementation"),
            lit(None).alias("ImplemenationStatus"),
            lit(None).alias("ConsiderControlProductionContracts"),
            lit(None).alias("ConsiderImplementationControl"),
            lit(None).alias("ResponsibleRegisterNotice"),
            lit(None).alias("ResponsibleCode"),
            lit(None).alias("RegisterNotice"),
            lit(None).alias("AssignedFlag"),
            lit(None).alias("AssignmentDate"),
            target_table_sb.Result.alias("Result"),
            lit(None).alias("ResultDate"),
            lit(None).alias("TerminationReason"),
            lit(None).alias("ApplicationImplementationFlag"),
            lit(None).alias("ApplicationImplementationDate"),
            lit(None).alias("Exposures"),
            lit(None).alias("ImplementationConsideration"),
            lit(None).alias("GalaxyFlag"),
            lit(None).alias("GalaxySecuritization"),
            lit(None).alias("Covid19"),
            lit(None).alias("Authority"),
            lit(None).alias("AuthorityDynamicChangedFlag"),
            target_table_sb.BusinessDate.alias("BusinessDate"),
        )

        # ================ Union ======================

        report_unified_df = result_df_cl.union(result_df_ml).union(result_df_sb)

        # ================ Append ======================

        report_unified_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_report_unified_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)